# Data Preprocessing & Model Training Pipeline
This notebook serves as the core environment for the **3 Trainers and 3 Models** AI purchase cost project.
Because the structure of `preprocessing.py` was robust, it has been integrated directly into this first step notebook so you can experiment efficiently!


In [ ]:
import pandas as pd
import numpy as np
import os
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split

# File path to the original raw dataset
DATA_PATH = r"data/AI purchase cost project (STUDENT DATA) V2 (1)(Blad2).csv"

# Let's load the data
df = pd.read_csv(DATA_PATH, sep=";", decimal=",")
print(f"Dataset Loaded Successfully! Shape: {df.shape}")
df.head()


## Step 1: Preprocessing Pipeline
We must convert dates, manage our high-cardinality discrete location codes (via Frequency Encoding to respect memory footprints), boolean cast string columns, and Target Encode our `Price category` classifications.


In [ ]:
def preprocess_data(df):
    print("Starting preprocessing...")
    df_clean = df.copy()

    # 1. Drop Shipments
    if 'Shipment' in df_clean.columns:
        df_clean = df_clean.drop('Shipment', axis=1)

    # 2. Date parsing (LAADDATUM)
    df_clean['LAADDATUM'] = pd.to_datetime(df_clean['LAADDATUM'], format='%d-%m-%Y', errors='coerce')
    df_clean['Load_Year'] = df_clean['LAADDATUM'].dt.year
    df_clean['Load_Month'] = df_clean['LAADDATUM'].dt.month
    df_clean['Load_DayOfWeek'] = df_clean['LAADDATUM'].dt.dayofweek
    df_clean['Load_Day'] = df_clean['LAADDATUM'].dt.day
    df_clean = df_clean.drop('LAADDATUM', axis=1)

    # 3. Y/N -> 1/0
    bool_cols = ['Crossdock', 'ADR', 'Express', 'Thermo']
    for col in bool_cols:
        df_clean[col] = df_clean[col].map({'Y': 1, 'N': 0}).fillna(0).astype('int8')

    # 4. Handle High-Cardinality Variables (Frequency Encoding)
    high_card_cols = ['Load code', 'Unload code', 'Distribution driven by code']
    for col in high_card_cols:
        freq_encoding = df_clean[col].value_counts() / len(df_clean)
        df_clean[col + '_freq'] = df_clean[col].map(freq_encoding)
        df_clean = df_clean.drop(col, axis=1)

    return df_clean

df_processed = preprocess_data(df)
print(f"Data preprocessed! New Shape: {df_processed.shape}")
df_processed.head()


## Step 2: Target Variable and Train-Test Split
Let's properly map and organize the target (`Price category`), then slice it into our Training and Testing sets.


In [ ]:
# Encode target
le = LabelEncoder()
df_processed['Price category encoded'] = le.fit_transform(df_processed['Price category'].astype(str))

# Create mapping dictionary for future reference
price_mapping = dict(zip(le.classes_, le.transform(le.classes_)))
print(f"Category Mapping (first 5 shown): {list(price_mapping.items())[:5]}")

# Define Features (X) and Target (y)
y = df_processed['Price category encoded']
X = df_processed.drop(['Price category', 'Price category encoded'], axis=1)

# Split data (80% Train, 20% validation)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f"Training features shape: {X_train.shape}")
print(f"Testing features shape: {X_test.shape}")


## Step 3: Setting Up Models (Trainer 1)
Here is where we will proceed with the first of our 3 Machine Learning models! A great baseline classifier for this task is a **Random Forest Classifier**.


In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

# Initialize the first model
rf_model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)

# TODO: Fit the model using rf_model.fit(X_train, y_train) 
# Note: This dataframe is extremely large (~600k rows) so training might take a minute!
